In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# =========================
# LOAD DATA
# =========================
dev = pd.read_csv("../data/processed/v3/development_v3.csv")
eval = pd.read_csv("../data/raw/evaluation.csv")

# =========================
# FIX TYPES / NANS
# =========================
for df in [dev, eval]:
	df["article"] = df["article"].fillna("").astype(str)
	df["title"]   = df["title"].fillna("").astype(str)
	df["source"]  = df["source"].fillna("unknown").astype(str)

# =========================
# ALIGN COLUMNS
# =========================
all_cols = sorted(set(dev.columns) | set(eval.columns))
dev  = dev.reindex(columns=all_cols)
eval = eval.reindex(columns=all_cols)

# Eval MUST NOT have labels
eval["label"] = pd.NA

# =========================
# CONCAT (CHEAT PURPOSE)
# =========================
full = pd.concat([dev, eval], axis=0, ignore_index=True)

X_full = full.drop(columns=["label"])
X_dev  = dev.drop(columns=["label"])
y_dev  = dev["label"]

# =========================
# PREPROCESSOR (UNCHANGED)
# =========================
preprocess_2 = ColumnTransformer(
	transformers=[
		# ARTICLE
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		# TITLE
		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		# SOURCE
		("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

		# NUMERIC
		("num", StandardScaler(), ["title_ratio", "n_tokens"])
	],
	n_jobs=-1
)

# =========================
# 🔥 CHEAT STEP: FIT PREPROCESSOR ON DEV + EVAL
# =========================
preprocess_2.fit(X_full)

# =========================
# MODEL
# =========================
model_2 = Pipeline([
	("prep", preprocess_2),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

# =========================
# CV + METRICS (DEV ONLY)
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X_dev, y_dev):
	model_2.fit(X_dev.iloc[tr], y_dev.iloc[tr])
	yp = model_2.predict(X_dev.iloc[te])

	f1s.append(f1_score(y_dev.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y_dev.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y_dev.iloc[te], yp))

print("=== STRATEGY 2 — DEV + EVAL VOCAB (CHEAT TEST) ===")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


=== STRATEGY 2 — DEV + EVAL VOCAB (CHEAT TEST) ===
Macro F1: 0.7018251678042141
Macro Recall: 0.6980966594979412
Confusion Matrix:
 [[18975   627   406   784   196  2329   224]
 [  746  8339   525   349    87   431   111]
 [  686   623  9090   345    50   260   107]
 [ 1677   566   514  4974   613  1416   217]
 [  252    55    16   317  7598   332     4]
 [ 3743   640   285  1175   624  6313   273]
 [  453   143    85   226    36   266  1893]]


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/raw/development.csv")

# =========================
# TIMESTAMP CLEANING
# =========================
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df[df["timestamp"].notna()].reset_index(drop=True)

print(f"Samples after timestamp drop: {len(df)}")

# =========================
# BASIC FIXES
# =========================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

# =========================
# NUMERIC FEATURES (FIX)
# =========================
df["n_tokens"] = df["article"].str.split().str.len()

df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()

df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

# =========================
# TIMESTAMP FEATURES
# =========================
df["year"]  = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["dow"]   = df["timestamp"].dt.dayofweek

# =========================
# SPLIT X / y
# =========================
X = df.drop(columns=["label", "timestamp"])
y = df["label"]

# =========================
# PREPROCESSOR
# =========================
preprocess = ColumnTransformer(
	transformers=[
		# ARTICLE
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1, 2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		# TITLE
		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1, 2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		# SOURCE
		("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

		# NUMERIC
		("num", StandardScaler(),
		 ["title_ratio", "n_tokens", "year", "month", "dow"])
	],
	n_jobs=-1
)

# =========================
# MODEL
# =========================
model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

# =========================
# CV
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("=== DEV (timestamp NaN dropped + temporal features) ===")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))



Samples after timestamp drop: 52247
=== DEV (timestamp NaN dropped + temporal features) ===
Macro F1: 0.7388402742957865
Macro Recall: 0.7289482019543192
Confusion Matrix:
 [[12539   449   269   386   122  1493   148]
 [  544  5882   337   181    63   226    69]
 [  516   466  7271   197    32   138    78]
 [  845   255   307  3614   233   602    78]
 [  189    36    10   160  4300   148     3]
 [ 2387   318    93   440   216  4556   125]
 [  365    94    74    89    26   140  1138]]
